In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,LabelEncoder, OneHotEncoder
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

In [2]:
#Load the trained model, scaler and geography encoder
model = load_model('models/churn_model.keras')
scaler = pickle.load(open('pickle/scaler.pkl', 'rb'))
geo_encoder = pickle.load(open('pickle/onehot_encoder_geo.pkl', 'rb'))

2026-07-04 18:05:06.423747: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-07-04 18:05:06.423771: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-04 18:05:06.423777: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-07-04 18:05:06.423788: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-04 18:05:06.423798: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
#Example input for a single customer
input_data = {
    'CreditScore': 619,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 42,
    'Tenure': 2,
    'Balance': 0.00,
    'NumOfProducts': 1,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 101348.88
}

input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.0,1,1,1,101348.88


In [4]:
#Encode Gender the same way as training (Male: 1, Female: 0)
input_df['Gender'] = input_df['Gender'].map({'Male': 1, 'Female': 0})

#One-hot encode Geography using the fitted encoder from training
geo_encoded = geo_encoder.transform(input_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_encoder.get_feature_names_out(['Geography']), index=input_df.index)

input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)

#Match the exact column order used to fit the scaler
feature_columns = ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
                    'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain']
input_df = input_df[feature_columns]
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain
0,619,0,42,2,0.0,1,1,1,101348.88,0.0,0.0


In [5]:
#Scale features and predict
input_scaled = scaler.transform(input_df)
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]

print(f"Churn probability: {prediction_proba:.4f}")
print("Prediction: Customer will churn" if prediction_proba > 0.5 else "Prediction: Customer will not churn")

2026-07-04 18:05:06.678040: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 509ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 515ms/step


Churn probability: 0.2574
Prediction: Customer will not churn
